# Estructuras de datos probabilísticas · Bloom, Count-Min y HyperLogLog

Estructuras que contestan preguntas sobre conjuntos y flujos enormes con muy poca memoria a cambio de un error acotado y conocido: el **filtro de Bloom** (¿está este elemento?, con falsos positivos pero sin falsos negativos), su variante **con contadores** (que permite borrar), el **Count-Min sketch** (¿cuántas veces ha aparecido?, sin quedarse nunca corto) y **HyperLogLog** (¿cuántos elementos distintos hay?). Cada ejercicio mide el error real frente a la fórmula teórica y frente al resultado exacto calculado con `set` o `Counter`. Las funciones hash independientes se obtienen de `hashlib.blake2b` con una sal distinta para cada una.

## 1. Funciones hash independientes con sal

Escribir `hash64(x, sal)`, que devuelve 64 bits de `blake2b` usando la sal para obtener funciones distintas, y `huella(x, sal)`, que devuelve dos valores de 64 bits de un solo cálculo (para el doble hashing de los ejercicios siguientes). Comprobar con la prueba chi-cuadrado que reparten 100 000 claves uniformemente en 64 cubetas, que las cubetas de dos sales distintas son independientes (tabla de contingencia 8 × 8) y que `hash()` de Python no sirve aquí: con enteros es la identidad y los múltiplos de 64 caen todos en la misma cubeta.

In [1]:
import hashlib
import math
from collections import Counter
from functools import cache

import numpy as np
from scipy import stats


def hash64(x, sal=0):
    datos = hashlib.blake2b(str(x).encode(), digest_size=8, salt=sal.to_bytes(16, "little")).digest()
    return int.from_bytes(datos, "little")


@cache
def huella(x, sal=0):
    datos = hashlib.blake2b(str(x).encode(), digest_size=16, salt=sal.to_bytes(16, "little")).digest()
    # h2 impar: así h1 + i·h2 no se queda quieto si m es potencia de 2
    return int.from_bytes(datos[:8], "little"), int.from_bytes(datos[8:], "little") | 1


claves = [f"isla-{i}" for i in range(100_000)]
for sal in range(3):
    cubetas = np.bincount([hash64(c, sal) % 64 for c in claves], minlength=64)
    p = stats.chisquare(cubetas).pvalue
    print(f"sal {sal}: cubetas entre {cubetas.min()} y {cubetas.max()} (esperado {len(claves) / 64:.0f}), p = {p:.3f}")
    assert p > 0.001

tabla = np.zeros((8, 8), dtype=int)
for c in claves:
    tabla[hash64(c, 0) % 8, hash64(c, 1) % 8] += 1
p_independencia = stats.chi2_contingency(tabla).pvalue
print(f"Independencia entre las sales 0 y 1: p = {p_independencia:.3f}")
assert p_independencia > 0.001

multiplos = [64 * i for i in range(10_000)]
con_hash = np.bincount([hash(x) % 64 for x in multiplos], minlength=64)
con_blake = np.bincount([hash64(x) % 64 for x in multiplos], minlength=64)
print(f"Múltiplos de 64 con hash(): {con_hash[0]} en la cubeta 0; con blake2b: máximo {con_blake.max()} por cubeta")
assert hash(12345) == 12345 and con_hash[0] == 10_000 and stats.chisquare(con_blake).pvalue > 0.001

sal 0: cubetas entre 1475 y 1651 (esperado 1562), p = 0.621
sal 1: cubetas entre 1471 y 1644 (esperado 1562), p = 0.913
sal 2: cubetas entre 1461 y 1647 (esperado 1562), p = 0.435


Independencia entre las sales 0 y 1: p = 0.230
Múltiplos de 64 con hash(): 10000 en la cubeta 0; con blake2b: máximo 185 por cubeta


## 2. Filtro de Bloom

Implementar `FiltroBloom(m, k)` sobre un `bytearray` de `m` bits: `añadir` pone a 1 los `k` bits que indican las funciones hash y `in` comprueba que estén todos a 1. Las `k` posiciones se calculan con doble hashing, $g_i(x) = h_1(x) + i\,h_2(x) \bmod m$, que con dos valores hash se comporta como `k` funciones independientes. Insertar 5000 códigos de reserva y comprobar que no hay falsos negativos y cuántos falsos positivos da con 20 000 códigos que no están.

In [2]:
class FiltroBloom:
    def __init__(self, m, k, sal=0, independientes=False):
        self.m, self.k, self.sal = m, k, sal
        self.independientes = independientes
        self.bits = bytearray((m + 7) // 8)

    def posiciones(self, x):
        if self.independientes:
            return [hash64(x, self.sal + i) % self.m for i in range(self.k)]
        h1, h2 = huella(x, self.sal)
        return [(h1 + i * h2) % self.m for i in range(self.k)]

    def añadir(self, x):
        for p in self.posiciones(x):
            self.bits[p >> 3] |= 1 << (p & 7)

    def __contains__(self, x):
        return all(self.bits[p >> 3] >> (p & 7) & 1 for p in self.posiciones(x))

    def bits_a_uno(self):
        return sum(b.bit_count() for b in self.bits)


reservas = [f"reserva-{i:06d}" for i in range(5000)]
ausentes = [f"reserva-{i:06d}" for i in range(500_000, 520_000)]

filtro = FiltroBloom(m=48_000, k=7)
for r in reservas:
    filtro.añadir(r)
falsos_positivos = sum(a in filtro for a in ausentes)
print(f"Memoria: {len(filtro.bits)} bytes para {len(reservas)} reservas ({filtro.m / len(reservas):.1f} bits por elemento)")
print(f"Bits a 1: {filtro.bits_a_uno() / filtro.m:.1%}")
print(f"Falsos positivos: {falsos_positivos} de {len(ausentes)} ({falsos_positivos / len(ausentes):.2%})")
assert all(r in filtro for r in reservas)
assert falsos_positivos / len(ausentes) < 0.02

Memoria: 6000 bytes para 5000 reservas (9.6 bits por elemento)
Bits a 1: 51.8%
Falsos positivos: 206 de 20000 (1.03%)


## 3. Dimensionado óptimo

Para `n` elementos y una tasa de falsos positivos objetivo `p`, el tamaño óptimo es $m = -\dfrac{n \ln p}{(\ln 2)^2}$ bits con $k = \dfrac{m}{n}\ln 2$ funciones hash, y la tasa teórica es $\left(1 - e^{-kn/m}\right)^k$. Escribir `dimensionar(n, p)` y mostrar una tabla para varios `n` y `p` con la memoria y los bits por elemento (que no dependen de `n`). Comprobar que el `k` de la fórmula es el que minimiza la tasa teórica buscando por fuerza bruta entre 1 y 40.

In [3]:
def dimensionar(n, p):
    m = math.ceil(-n * math.log(p) / math.log(2) ** 2)
    return m, max(1, round(m / n * math.log(2)))


def fp_teorica(m, n, k):
    return (1 - math.exp(-k * n / m)) ** k


print(f"{'n':>9} {'p':>7} {'m (bits)':>11} {'memoria':>10} {'k':>3} {'bits/elem':>10} {'p real':>9}")
for n in (10_000, 1_000_000):
    for p in (0.01, 0.001, 0.0001):
        m, k = dimensionar(n, p)
        k_bruta = min(range(1, 41), key=lambda j: fp_teorica(m, n, j))
        print(f"{n:9} {p:7} {m:11} {m / 8 / 1024:8.1f} KiB {k:3} {m / n:10.2f} {fp_teorica(m, n, k):9.6f}")
        assert k == k_bruta and fp_teorica(m, n, k) <= p * 1.01

        n       p    m (bits)    memoria   k  bits/elem    p real
    10000    0.01       95851     11.7 KiB   7       9.59  0.010039
    10000   0.001      143776     17.6 KiB  10      14.38  0.001000
    10000  0.0001      191702     23.4 KiB  13      19.17  0.000100
  1000000    0.01     9585059   1170.1 KiB   7       9.59  0.010039
  1000000   0.001    14377588   1755.1 KiB  10      14.38  0.001000
  1000000  0.0001    19170117   2340.1 KiB  13      19.17  0.000100


## 4. Tasa de falsos positivos medida frente a la teórica

Dimensionar un filtro para 5000 elementos con `p = 1 %` y llenarlo poco a poco hasta el doble de su capacidad. En cada etapa, medir la tasa de falsos positivos con 20 000 elementos ausentes y compararla con la teórica: la diferencia debe caber en el margen del error de muestreo binomial. Ver cómo se dispara al pasarse de capacidad.

In [4]:
m, k = dimensionar(5000, 0.01)
filtro = FiltroBloom(m, k)
todas = [f"reserva-{i:06d}" for i in range(10_000)]
print(f"m = {m} bits, k = {k}")
print(f"{'insertados':>10} {'medida':>8} {'teórica':>8} {'bits a 1':>9}")
insertados = 0
for objetivo in (1000, 2500, 5000, 7500, 10_000):
    for r in todas[insertados:objetivo]:
        filtro.añadir(r)
    insertados = objetivo
    medida = sum(a in filtro for a in ausentes) / len(ausentes)
    teorica = fp_teorica(m, insertados, k)
    print(f"{insertados:10} {medida:8.4f} {teorica:8.4f} {filtro.bits_a_uno() / m:9.1%}")
    assert abs(medida - teorica) <= 4 * math.sqrt(teorica * (1 - teorica) / len(ausentes)) + 1e-4

m = 47926 bits, k = 7
insertados   medida  teórica  bits a 1
      1000   0.0000   0.0000     13.5%
      2500   0.0001   0.0003     30.5%
      5000   0.0097   0.0100     51.7%
      7500   0.0586   0.0579     66.6%
     10000   0.1592   0.1574     77.0%


## 5. Efecto del número de funciones hash

Con `m` y `n` fijos, probar `k` de 1 a 12. Con pocas funciones cada consulta mira pocos bits; con muchas, el filtro se llena de unos. Medir la tasa para cada `k`, comprobar que sigue a la teórica y que el mínimo medido cae cerca del `k` óptimo, en una zona donde la curva es casi plana.

In [5]:
m, k_optimo = dimensionar(5000, 0.01)
medidas = {}
print(f"{'k':>3} {'medida':>8} {'teórica':>8}")
for k in range(1, 13):
    f = FiltroBloom(m, k)
    for r in reservas:
        f.añadir(r)
    medidas[k] = sum(a in f for a in ausentes) / len(ausentes)
    teorica = fp_teorica(m, len(reservas), k)
    print(f"{k:3} {medidas[k]:8.4f} {teorica:8.4f} {'←' if k == k_optimo else ''}")
    assert abs(medidas[k] - teorica) <= 4 * math.sqrt(teorica * (1 - teorica) / len(ausentes)) + 1e-4
k_medido = min(medidas, key=medidas.get)
print(f"Mínimo medido en k = {k_medido}; óptimo teórico k = {k_optimo}")
assert abs(k_medido - k_optimo) <= 2

  k   medida  teórica
  1   0.1003   0.0991 
  2   0.0355   0.0355 
  3   0.0209   0.0194 
  4   0.0140   0.0136 
  5   0.0107   0.0111 


  6   0.0090   0.0101 
  7   0.0097   0.0100 ←
  8   0.0104   0.0105 
  9   0.0112   0.0115 


 10   0.0123   0.0130 
 11   0.0154   0.0150 
 12   0.0185   0.0176 
Mínimo medido en k = 6; óptimo teórico k = 7


## 6. `k` funciones con sal frente a doble hashing

Comparar el filtro con doble hashing (un solo `blake2b` de 128 bits por elemento) con uno que calcula `k` hash independientes con `k` sales distintas. Repetir con 5 sales de partida y comprobar que la tasa media de falsos positivos es prácticamente la misma, mientras que el número de llamadas a `blake2b` se divide por `k`.

In [6]:
m, k = dimensionar(5000, 0.01)
tasas = {"doble hashing": [], "k sales": []}
for semilla in range(5):
    for nombre, independientes in (("doble hashing", False), ("k sales", True)):
        f = FiltroBloom(m, k, sal=100 * semilla, independientes=independientes)
        for r in reservas:
            f.añadir(r)
        tasas[nombre].append(sum(a in f for a in ausentes) / len(ausentes))
for nombre, valores in tasas.items():
    llamadas = len(reservas) + len(ausentes) if nombre == "doble hashing" else k * (len(reservas) + len(ausentes))
    print(f"{nombre:14} tasa media {np.mean(valores):.4f} (±{np.std(valores):.4f}), llamadas a blake2b por filtro: {llamadas}")
print(f"Teórica: {fp_teorica(m, len(reservas), k):.4f}")
assert abs(np.mean(tasas["doble hashing"]) - np.mean(tasas["k sales"])) < 0.002

doble hashing  tasa media 0.0102 (±0.0006), llamadas a blake2b por filtro: 25000
k sales        tasa media 0.0098 (±0.0004), llamadas a blake2b por filtro: 175000
Teórica: 0.0100


## 7. Unión, intersección y estimación del tamaño

Dos filtros con los mismos `m`, `k` y sal se unen con un OR bit a bit, y el resultado es exactamente el filtro de la unión. Además, el número `X` de bits a 1 permite estimar cuántos elementos tiene un filtro: $n \approx -\frac{m}{k}\ln\left(1 - \frac{X}{m}\right)$. Comprobar la unión con dos conjuntos de reservas solapados, estimar sus tamaños y el de la intersección por inclusión-exclusión, y compararlos con los exactos.

In [7]:
def estimar_n(f):
    return -f.m / f.k * math.log(1 - f.bits_a_uno() / f.m)


def construir(elementos, m, k):
    f = FiltroBloom(m, k)
    for x in elementos:
        f.añadir(x)
    return f


m, k = dimensionar(10_000, 0.01)
a = {f"reserva-{i:06d}" for i in range(0, 3000)}
b = {f"reserva-{i:06d}" for i in range(2000, 6000)}
fa, fb = construir(a, m, k), construir(b, m, k)
union = FiltroBloom(m, k)
union.bits = bytearray(x | y for x, y in zip(fa.bits, fb.bits))
assert union.bits == construir(a | b, m, k).bits

estimaciones = {"A": estimar_n(fa), "B": estimar_n(fb), "A ∪ B": estimar_n(union)}
estimaciones["A ∩ B"] = estimaciones["A"] + estimaciones["B"] - estimaciones["A ∪ B"]
exactos = {"A": len(a), "B": len(b), "A ∪ B": len(a | b), "A ∩ B": len(a & b)}
for nombre in exactos:
    error = estimaciones[nombre] / exactos[nombre] - 1
    print(f"{nombre:6} exacto {exactos[nombre]:5}  estimado {estimaciones[nombre]:8.1f}  error {error:+.1%}")
    assert abs(error) < (0.2 if nombre == "A ∩ B" else 0.05)

A      exacto  3000  estimado   2999.2  error -0.0%
B      exacto  4000  estimado   3991.6  error -0.2%
A ∪ B  exacto  6000  estimado   6000.2  error +0.0%
A ∩ B  exacto  1000  estimado    990.5  error -0.9%


## 8. Bloom con contadores: permitir borrar

En un filtro de Bloom normal no se puede borrar: poner a 0 los bits de un elemento puede borrar también bits de otros, y aparecen falsos negativos. Implementar `BloomContador`, que guarda en cada posición un contador de 4 bits (de 0 a 15, que se queda fijo si se satura) en lugar de un bit. Insertar 5000 reservas, borrar la mitad y comparar con el «borrado» ingenuo en un Bloom normal: falsos negativos, tasa de falsos positivos antes y después, valor máximo de los contadores y memoria.

In [8]:
class BloomContador(FiltroBloom):
    MAXIMO = 15

    def __init__(self, m, k, sal=0):
        super().__init__(m, k, sal)
        self.contadores = bytearray(m)

    def añadir(self, x):
        for p in self.posiciones(x):
            self.contadores[p] = min(self.MAXIMO, self.contadores[p] + 1)

    def borrar(self, x):
        # solo es correcto borrar lo que se añadió
        for p in self.posiciones(x):
            if self.contadores[p] < self.MAXIMO:
                self.contadores[p] -= 1

    def __contains__(self, x):
        return all(self.contadores[p] for p in self.posiciones(x))


def tasa_fp(f):
    return sum(a in f for a in ausentes) / len(ausentes)


m, k = dimensionar(5000, 0.01)
normal, contador = FiltroBloom(m, k), BloomContador(m, k)
for r in reservas:
    normal.añadir(r)
    contador.añadir(r)
fp_antes = tasa_fp(contador)
assert fp_antes == tasa_fp(normal)

borradas, quedan = reservas[:2500], reservas[2500:]
for r in borradas:
    contador.borrar(r)
    for p in normal.posiciones(r):
        normal.bits[p >> 3] &= ~(1 << (p & 7))

print(f"{'':22} {'Bloom normal':>13} {'con contadores':>15}")
print(f"{'falsos negativos':22} {sum(r not in normal for r in quedan):13} {sum(r not in contador for r in quedan):15}")
print(f"{'FP antes de borrar':22} {fp_antes:13.4f} {fp_antes:15.4f}")
print(f"{'FP después':22} {tasa_fp(normal):13.4f} {tasa_fp(contador):15.4f}")
print(f"{'memoria (bytes)':22} {m // 8:13} {m // 2:15}")
print(f"Contador máximo: {max(contador.contadores)}; teórica con 2500 elementos: {fp_teorica(m, 2500, k):.4f}")
assert all(r in contador for r in quedan)
assert sum(r not in normal for r in quedan) > 0
assert max(contador.contadores) < BloomContador.MAXIMO

                        Bloom normal  con contadores
falsos negativos                2319               0
FP antes de borrar            0.0097          0.0097
FP después                    0.0000          0.0003
memoria (bytes)                 5990           23963
Contador máximo: 5; teórica con 2500 elementos: 0.0003


## 9. Count-Min sketch

Un Count-Min sketch es una matriz de `d` filas por `w` columnas de contadores; cada fila tiene su propia función hash (su propia sal), añadir suma 1 en una celda de cada fila y la estimación es el mínimo de esas `d` celdas. Nunca se queda corto y, con $w = \lceil e/\varepsilon \rceil$ y $d = \lceil \ln(1/\delta) \rceil$, se pasa como mucho en $\varepsilon N$ con probabilidad $1 - \delta$. Generar un flujo de 200 000 términos de búsqueda con distribución de Zipf ($s = 1{,}1$, 10 000 términos) y comparar las estimaciones con `Counter`.

In [9]:
# las filas usan sales distintas: con doble hashing, dos términos que coinciden en h1 y h2 módulo w chocarían en todas
hash_fila = cache(hash64)


class CountMin:
    def __init__(self, w, d, sal=0, conservador=False):
        self.w, self.d, self.sal = w, d, sal
        self.conservador = conservador
        self.filas = [[0] * w for _ in range(d)]
        self.total = 0

    @classmethod
    def para(cls, epsilon, delta, **opciones):
        return cls(math.ceil(math.e / epsilon), math.ceil(math.log(1 / delta)), **opciones)

    def columnas(self, x):
        return [hash_fila(x, self.sal + i) % self.w for i in range(self.d)]

    def añadir(self, x):
        self.total += 1
        columnas = self.columnas(x)
        if self.conservador:
            nuevo = min(fila[c] for fila, c in zip(self.filas, columnas)) + 1
            for fila, c in zip(self.filas, columnas):
                fila[c] = max(fila[c], nuevo)
        else:
            for fila, c in zip(self.filas, columnas):
                fila[c] += 1

    def estimar(self, x):
        return min(fila[c] for fila, c in zip(self.filas, self.columnas(x)))


rng = np.random.default_rng(33)
rangos = np.arange(1, 10_001)
probabilidades = rangos ** -1.1 / np.sum(rangos ** -1.1)
flujo = [f"termino-{r}" for r in rng.choice(rangos, size=200_000, p=probabilidades)]
exacto = Counter(flujo)

epsilon, delta = 0.001, 0.01
cm = CountMin.para(epsilon, delta)
for x in flujo:
    cm.añadir(x)
errores = {x: cm.estimar(x) - c for x, c in exacto.items()}
cota = epsilon * cm.total
dentro = sum(e <= cota for e in errores.values()) / len(errores)
print(f"w = {cm.w}, d = {cm.d}: {cm.w * cm.d} contadores para {len(exacto)} términos distintos")
print(f"{'término':14} {'real':>6} {'estimado':>9}")
for x, c in exacto.most_common(5):
    print(f"{x:14} {c:6} {cm.estimar(x):9}")
print(f"Error medio {np.mean(list(errores.values())):.2f}, máximo {max(errores.values())}, cota εN = {cota:.0f}")
print(f"Términos con error ≤ εN: {dentro:.2%} (garantía: ≥ {1 - delta:.0%})")
assert min(errores.values()) >= 0 and dentro >= 1 - delta

w = 2719, d = 5: 13595 contadores para 9024 términos distintos
término          real  estimado
termino-1       30232     30247
termino-2       14167     14167
termino-3        9250      9255
termino-4        6507      6517
termino-5        5192      5194
Error medio 4.65, máximo 65, cota εN = 200
Términos con error ≤ εN: 100.00% (garantía: ≥ 99%)


## 10. Anchura del sketch y actualización conservadora

Variar la anchura `w` con `d = 4` fijo y medir el error medio y el máximo frente a la cota $e N / w$. Comparar con la **actualización conservadora**, que al añadir solo sube las celdas que están en el mínimo (las demás ya sobreestiman): sigue sin quedarse nunca corta y reduce el error.

In [10]:
print(f"{'w':>5} {'cota eN/w':>10} {'medio':>7} {'máximo':>7} {'medio cons.':>12} {'máximo cons.':>13}")
for w in (100, 300, 1000, 3000):
    resultados = []
    for conservador in (False, True):
        sketch = CountMin(w, 4, sal=7, conservador=conservador)
        for x in flujo:
            sketch.añadir(x)
        errores = [sketch.estimar(x) - c for x, c in exacto.items()]
        assert min(errores) >= 0
        resultados.append((np.mean(errores), max(errores)))
    (medio, maximo), (medio_c, maximo_c) = resultados
    print(f"{w:5} {math.e * len(flujo) / w:10.0f} {medio:7.1f} {maximo:7} {medio_c:12.1f} {maximo_c:13}")
    assert medio_c <= medio

    w  cota eN/w   medio  máximo  medio cons.  máximo cons.


  100       5437   691.3    3796        404.7          3015


  300       1812   165.9    1207         93.5          1021


 1000        544    30.4     186         15.5           123


 3000        181     4.9      70          2.0            59


## 11. Términos más buscados con Count-Min

Encontrar en una sola pasada los términos que superan el 0,5 % del flujo (*heavy hitters*) sin guardar un contador por término: al procesar cada elemento, si su estimación supera el 0,5 % de lo visto hasta entonces, se guarda como candidato; al final se filtran los candidatos con el total. Como el sketch nunca se queda corto, no se escapa ninguno de los verdaderos; comprobarlo con `Counter` y medir cuántos candidatos sobran.

In [11]:
phi = 0.005
sketch = CountMin.para(0.001, 0.01, sal=11)
candidatos = set()
for x in flujo:
    sketch.añadir(x)
    if sketch.estimar(x) >= phi * sketch.total:
        candidatos.add(x)
encontrados = {x for x in candidatos if sketch.estimar(x) >= phi * sketch.total}
verdaderos = {x for x, c in exacto.items() if c >= phi * len(flujo)}
print(f"Candidatos guardados: {len(candidatos)}; tras filtrar: {len(encontrados)}; verdaderos: {len(verdaderos)}")
print("Sobran:", sorted(encontrados - verdaderos, key=lambda x: -exacto[x]))
assert verdaderos <= encontrados

Candidatos guardados: 126; tras filtrar: 22; verdaderos: 22
Sobran: []


## 12. HyperLogLog

Implementar HyperLogLog con $m = 2^b$ registros de un byte: los $b$ primeros bits del hash eligen el registro y en él se guarda la posición del primer 1 en el resto de bits (cuantos más ceros seguidos, más elementos distintos hacen falta para verlo). La estimación es $\alpha_m m^2 / \sum_j 2^{-M_j}$, con la corrección de *linear counting*, $m \ln(m/V)$, cuando la estimación es pequeña y quedan `V` registros a cero. Contar los visitantes distintos de un flujo de 300 000 visitas con 1 KiB de registros, y mostrar con varios tamaños por qué hace falta la corrección para cardinalidades pequeñas.

In [12]:
class HyperLogLog:
    def __init__(self, b, sal=0):
        self.b, self.m, self.sal = b, 1 << b, sal
        self.registros = bytearray(self.m)
        self.alfa = {16: 0.673, 32: 0.697, 64: 0.709}.get(self.m, 0.7213 / (1 + 1.079 / self.m))

    def añadir_hash(self, h):
        j = h >> (64 - self.b)
        resto = h & ((1 << (64 - self.b)) - 1)
        rho = (64 - self.b) - resto.bit_length() + 1
        if rho > self.registros[j]:
            self.registros[j] = rho

    def añadir(self, x):
        self.añadir_hash(hash64(x, self.sal))

    def estimacion_bruta(self):
        return self.alfa * self.m ** 2 / sum(2.0 ** -r for r in self.registros)

    def estimar(self):
        e = self.estimacion_bruta()
        ceros = self.registros.count(0)
        if e <= 2.5 * self.m and ceros:
            return self.m * math.log(self.m / ceros)
        return e


rng = np.random.default_rng(12)
visitas = [f"visitante-{v}" for v in rng.integers(0, 60_000, size=300_000)]
hll = HyperLogLog(b=10)
for v in visitas:
    hll.añadir(v)
distintos = len(set(visitas))
print(f"Visitas: {len(visitas)}; distintos exactos: {distintos}; estimados: {hll.estimar():.0f} "
      f"(error {hll.estimar() / distintos - 1:+.2%}) con {len(hll.registros)} bytes")
assert abs(hll.estimar() / distintos - 1) < 0.1

print(f"\n{'n':>7} {'bruta':>9} {'corregida':>10}")
for n in (10, 100, 1000, 10_000, 100_000):
    h = HyperLogLog(b=10, sal=1)
    for i in range(n):
        h.añadir(f"usuario-{i}")
    print(f"{n:7} {h.estimacion_bruta():9.0f} {h.estimar():10.0f}")
    assert abs(h.estimar() / n - 1) < 0.1

Visitas: 300000; distintos exactos: 59589; estimados: 62621 (error +5.09%) con 1024 bytes

      n     bruta  corregida
     10       742         10
    100       784         97
   1000      1333       1023
  10000      9946       9946
 100000     97830      97830


## 13. Error relativo según el número de registros

El error típico de HyperLogLog es $1{,}04/\sqrt{m}$. Medirlo: contar 20 000 elementos distintos con $b$ de 4 a 14 (de 16 a 16 384 registros) repitiendo con 12 sales distintas, y comparar la raíz del error cuadrático medio relativo con la fórmula. Los hash de cada sal se calculan una vez y sirven para todos los tamaños.

In [13]:
n = 20_000
elementos = [f"matricula-{i}" for i in range(n)]
hashes_por_sal = [[hash64(x, 1000 + s) for x in elementos] for s in range(12)]
print(f"{'b':>3} {'registros':>9} {'memoria':>9} {'error medido':>13} {'1,04/√m':>8}")
errores_por_b = {}
for b in (4, 6, 8, 10, 12, 14):
    errores = []
    for hashes in hashes_por_sal:
        h = HyperLogLog(b)
        for x in hashes:
            h.añadir_hash(x)
        errores.append(h.estimar() / n - 1)
    rmse = math.sqrt(np.mean(np.square(errores)))
    teorico = 1.04 / math.sqrt(1 << b)
    errores_por_b[b] = rmse
    print(f"{b:3} {1 << b:9} {1 << b:7} B {rmse:13.2%} {teorico:8.2%}")
    assert rmse < 2 * teorico
assert errores_por_b[14] < errores_por_b[8] < errores_por_b[4]

  b registros   memoria  error medido  1,04/√m
  4        16      16 B        30.06%   26.00%
  6        64      64 B        16.46%   13.00%
  8       256     256 B         8.54%    6.50%
 10      1024    1024 B         2.78%    3.25%


 12      4096    4096 B         1.66%    1.62%
 14     16384   16384 B         0.85%    0.81%


## 14. Unir HyperLogLog

Dos HyperLogLog con el mismo `b` y la misma sal se unen tomando el máximo registro a registro, y el resultado es idéntico al HyperLogLog de la unión, así que se pueden contar distintos en varios servidores y juntarlos después. Comprobarlo con los visitantes de dos islas que se solapan, y estimar la unión y la intersección (por inclusión-exclusión) frente a los valores exactos. La intersección tiene un error relativo mucho mayor porque es la resta de dos estimaciones grandes.

In [14]:
def hll_de(elementos, b=12, sal=5):
    h = HyperLogLog(b, sal)
    for x in elementos:
        h.añadir(x)
    return h


tenerife = {f"visitante-{i}" for i in range(0, 30_000)}
gran_canaria = {f"visitante-{i}" for i in range(20_000, 60_000)}
ht, hg = hll_de(tenerife), hll_de(gran_canaria)
union = HyperLogLog(12, 5)
union.registros = bytearray(map(max, ht.registros, hg.registros))
assert union.registros == hll_de(tenerife | gran_canaria).registros

estimados = {"Tenerife": ht.estimar(), "Gran Canaria": hg.estimar(), "unión": union.estimar()}
estimados["intersección"] = estimados["Tenerife"] + estimados["Gran Canaria"] - estimados["unión"]
exactos = {"Tenerife": len(tenerife), "Gran Canaria": len(gran_canaria),
           "unión": len(tenerife | gran_canaria), "intersección": len(tenerife & gran_canaria)}
for nombre in exactos:
    error = estimados[nombre] / exactos[nombre] - 1
    print(f"{nombre:13} exacto {exactos[nombre]:6}  estimado {estimados[nombre]:8.0f}  error {error:+.2%}")
    assert abs(error) < (0.25 if nombre == "intersección" else 0.05)

Tenerife      exacto  30000  estimado    30070  error +0.23%
Gran Canaria  exacto  40000  estimado    39715  error -0.71%
unión         exacto  60000  estimado    59267  error -1.22%
intersección  exacto  10000  estimado    10517  error +5.17%
